In [ ]:
import tensorflow as tf
import tensorflow.compat.v1 as tf_v1
from tensorflow.keras.datasets import mnist
import numpy as np

# 禁用 Eager Execution
tf_v1.disable_eager_execution()

# 加载 MNIST 数据集
(train_images, train_labels), (test_images, test_labels) = mnist.load_data()

# 数据预处理
train_images = train_images.reshape(-1, 784) / 255.0
test_images = test_images.reshape(-1, 784) / 255.0
train_labels = tf_v1.keras.utils.to_categorical(train_labels, 10)
test_labels = tf_v1.keras.utils.to_categorical(test_labels, 10)

# 替换原来的 mnist 数据加载部分
mnist = {
    'train': {'images': train_images, 'labels': train_labels},
    'test': {'images': test_images, 'labels': test_labels}
}

learning_rate = 1e-4
keep_prob_rate = 0.7
max_epoch = 2000

def compute_accuracy(v_xs, v_ys):
    global prediction
    y_pre = sess.run(prediction, feed_dict={xs: v_xs, keep_prob: 1})
    correct_prediction = tf_v1.equal(tf_v1.argmax(y_pre, 1), tf_v1.argmax(v_ys, 1))
    accuracy = tf_v1.reduce_mean(tf_v1.cast(correct_prediction, tf_v1.float32))
    result = sess.run(accuracy)
    return result

def weight_variable(shape):
    initial = tf_v1.truncated_normal(shape, stddev=0.1)
    return tf_v1.Variable(initial)

def bias_variable(shape):
    initial = tf_v1.constant(0.1, shape=shape)
    return tf_v1.Variable(initial)

def conv2d(x, W):
    return tf_v1.nn.conv2d(x, W, strides=[1, 1, 1, 1], padding='SAME')

def max_pool_2x2(x):
    return tf_v1.nn.max_pool(x, ksize=[1, 2, 2, 1], strides=[1, 2, 2, 1], padding='SAME')

# define placeholder for inputs to network
xs = tf_v1.placeholder(tf_v1.float32, [None, 784]) / 255.
ys = tf_v1.placeholder(tf_v1.float32, [None, 10])
keep_prob = tf_v1.placeholder(tf_v1.float32)
x_image = tf_v1.reshape(xs, [-1, 28, 28, 1])

# 卷积层 1
W_conv1 = weight_variable([5, 5, 1, 32])  # 使用 5x5 的卷积核
b_conv1 = bias_variable([32])
h_conv1 = tf_v1.nn.relu(conv2d(x_image, W_conv1) + b_conv1)
h_pool1 = max_pool_2x2(h_conv1)

# 卷积层 2
W_conv2 = weight_variable([5, 5, 32, 64])  # 使用 5x5 的卷积核
b_conv2 = bias_variable([64])
h_conv2 = tf_v1.nn.relu(conv2d(h_pool1, W_conv2) + b_conv2)
h_pool2 = max_pool_2x2(h_conv2)

# 全连接层 1
W_fc1 = weight_variable([7 * 7 * 64, 1024])
b_fc1 = bias_variable([1024])
h_pool2_flat = tf_v1.reshape(h_pool2, [-1, 7 * 7 * 64])
h_fc1 = tf_v1.nn.relu(tf_v1.matmul(h_pool2_flat, W_fc1) + b_fc1)
h_fc1_drop = tf_v1.nn.dropout(h_fc1, keep_prob)

# 全连接层 2
W_fc2 = weight_variable([1024, 10])
b_fc2 = bias_variable([10])
logits = tf_v1.matmul(h_fc1_drop, W_fc2) + b_fc2
prediction = tf_v1.nn.softmax(logits)

# 交叉熵损失函数
cross_entropy = tf_v1.reduce_mean(tf_v1.nn.softmax_cross_entropy_with_logits(logits=logits, labels=ys))
train_step = tf_v1.train.AdamOptimizer(learning_rate).minimize(cross_entropy)

with tf_v1.Session() as sess:
    init = tf_v1.global_variables_initializer()
    sess.run(init)

    for i in range(max_epoch):
        start = (i * 100) % len(mnist['train']['images'])
        end = ((i + 1) * 100) % len(mnist['train']['images'])
        if end <= start:
            continue  # 避免越界
        batch_xs, batch_ys = mnist['train']['images'][start:end], mnist['train']['labels'][start:end]
        sess.run(train_step, feed_dict={xs: batch_xs, ys: batch_ys, keep_prob: keep_prob_rate})
        if i % 100 == 0:
            print(compute_accuracy(
                mnist['test']['images'][:1000], mnist['test']['labels'][:1000]))


0.084
0.83
0.903
0.925
0.94
0.951
0.938
0.956
0.959
0.963
0.963
0.973
0.969
0.97
0.97
0.972
0.977
0.975
0.973
0.979
